# 3.2 — Segmentation des phrases

**Traitement automatique du langage naturel (TALN) | Python, spaCy et NLTK**

Ce notebook présente plusieurs méthodes pour découper un texte en phrases, examiner les frontières entre phrases et adapter la segmentation à des besoins particuliers.

## Objectifs pédagogiques

À la fin de ce notebook, vous serez capable de :
- expliquer la segmentation des phrases ;
- utiliser `spaCy` et `NLTK` pour segmenter un texte ;
- inspecter les frontières de phrases dans `spaCy` ;
- comprendre pourquoi la ponctuation seule ne suffit pas toujours ;
- tester la segmentation sur des textes anglais et arabes.

## 1. Préparation de l’environnement

Exécutez cette cellule si les bibliothèques ne sont pas encore installées. Dans Google Colab, vous pouvez l’exécuter directement.

In [ ]:
%pip install -q spacy nltk

## 2. Charger le modèle anglais de spaCy

Le modèle `en_core_web_sm` fournit un analyseur syntaxique qui peut déterminer les frontières entre phrases. Si le modèle n’est pas installé, la cellule suivante le télécharge.

In [ ]:
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("Pipeline:", nlp.pipe_names)

## 3. Segmenter un texte avec spaCy

In [ ]:
text = (
    "This is the first sentence. "
    "This is another sentence. "
    "This is the last sentence."
)

doc = nlp(text)

for i, sent in enumerate(doc.sents, start=1):
    print(f"Sentence {i}: {sent.text}")

**À retenir :** `nlp(text)` crée un objet `Doc`. L’attribut `doc.sents` permet d’itérer sur les segments reconnus comme phrases.

## 4. Examiner les objets `Span`

In [ ]:
sentences = list(doc.sents)

print("Number of sentences:", len(sentences))
print("Second sentence:", sentences[1].text)
print("Token interval [start, end):", sentences[1].start, sentences[1].end)
print("Object type:", type(sentences[1]))

Une phrase est représentée par un objet `Span`. Les indices `start` et `end` délimitent les tokens dans le document ; `end` est exclusif.

## 5. Repérer le début des phrases

In [ ]:
text = "This is a sentence. That is another sentence. Here is a third."
doc = nlp(text)

for token in doc:
    print(f"{token.text:>10} | is_sent_start={token.is_sent_start}")

`token.is_sent_start` vaut `True` pour le premier token d’une phrase et `False` pour les autres tokens. Le premier token du document est généralement un début de phrase.

## 6. Pourquoi la segmentation peut être difficile

Un point peut terminer une phrase, mais il peut aussi apparaître dans une abréviation, une adresse web ou un nombre décimal. Les modèles linguistiques utilisent le contexte pour mieux décider.

In [ ]:
examples = [
    "Hello Mr. Smith, how are you today?",
    "The value is 3.26 dollars. The meeting is on Monday.",
    "Visit example.com for more information."
]

for text in examples:
    print("\nTEXT:", text)
    for sent in nlp(text).sents:
        print(" -", sent.text)

## 7. Segmentation personnalisée avec spaCy

Il est possible d’ajouter une règle métier. Exemple : considérer un point-virgule (`;`) comme une frontière de phrase. Le composant ci-dessous est ajouté **avant** l’analyseur syntaxique afin que celui-ci puisse tenir compte de la frontière.

In [ ]:
from spacy.language import Language

@Language.component("semicolon_sentence_boundary")
def semicolon_sentence_boundary(doc):
    for token in doc[:-1]:
        if token.text == ";":
            doc[token.i + 1].is_sent_start = True
    return doc

# Add the component only once.
if "semicolon_sentence_boundary" not in nlp.pipe_names:
    nlp.add_pipe("semicolon_sentence_boundary", before="parser")

custom_text = "Management is doing things right; leadership is doing the right things."
custom_doc = nlp(custom_text)

for sent in custom_doc.sents:
    print("-", sent.text)

**Remarque :** les règles personnalisées doivent être évaluées sur des exemples représentatifs. Une frontière utile dans un cas peut produire une segmentation incorrecte dans un autre.

## 8. Utiliser NLTK

NLTK propose `sent_tokenize`, un outil pratique pour découper un texte en phrases. Il peut nécessiter le téléchargement de ressources de tokenisation.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

for resource in ("punkt", "punkt_tab"):
    try:
        nltk.download(resource, quiet=True)
    except Exception as exc:
        print(f"Could not download {resource}: {exc}")

example_text = (
    "Hello Mr. Smith, how are you doing today? "
    "The weather is great, and Python is awesome. "
    "The sky is pinkish-blue. You shouldn't eat cardboard."
)

for i, sentence in enumerate(sent_tokenize(example_text), start=1):
    print(f"{i}. {sentence}")

## 9. Comparer spaCy et NLTK

In [ ]:
text = (
    "Dr. Smith arrived at 10.30 a.m. "
    "He gave a short talk. Everyone enjoyed it!"
)

print("spaCy:")
for sent in nlp(text).sents:
    print("-", sent.text)

print("\nNLTK:")
for sent in sent_tokenize(text):
    print("-", sent)

Les deux bibliothèques peuvent produire des résultats différents selon le texte, les modèles et les ressources utilisés. Pour un projet réel, examinez les cas limites de votre corpus.

## 10. Texte arabe : points d’attention

Le modèle anglais `en_core_web_sm` n’est pas adapté à l’analyse linguistique de l’arabe. Pour illustrer une segmentation simple, `spacy.blank('ar')` fournit un tokenizer arabe sans analyseur syntaxique. Les frontières de phrases doivent être ajoutées explicitement, par exemple avec `sentencizer`.

In [ ]:
arabic_nlp = spacy.blank("ar")
arabic_nlp.add_pipe("sentencizer")

arabic_text = "هذه هي الجملة الأولى. هذه هي الجملة الثانية؟ وهذه هي الجملة الثالثة!"
arabic_doc = arabic_nlp(arabic_text)

for i, sent in enumerate(arabic_doc.sents, start=1):
    print(f"{i}. {sent.text}")

Cette démonstration dépend de la ponctuation. Pour une analyse morphologique ou syntaxique de l’arabe, choisissez un modèle et des ressources adaptés à la variété linguistique visée.

## 11. Découpage simple avec `str.split()`

La méthode Python `split()` sépare généralement sur les espaces. Elle ne tient pas compte de la structure linguistique et conserve souvent la ponctuation attachée aux mots.

In [ ]:
text = "Sentence one. Sentence two! Is this sentence three?"
print(text.split())

## 12. Exercices

1. Écrivez un texte contenant au moins quatre phrases et affichez chaque phrase avec `spaCy`.
2. Comparez les résultats de `spaCy` et `NLTK` sur une phrase contenant une abréviation.
3. Testez le texte `I met Prof. Mohamed yesterday. We discussed NLP.` et observez la segmentation.
4. Segmentez un court texte arabe avec `spacy.blank('ar')` et `sentencizer`.
5. Expliquez pourquoi `str.split()` n’est pas une méthode fiable pour segmenter les phrases.

## 13. Résumé

- La segmentation des phrases identifie les unités phrastiques d’un texte.
- `spaCy` utilise ici un modèle anglais avec analyseur syntaxique ; `doc.sents` renvoie les phrases.
- `NLTK.sent_tokenize` propose une autre approche de segmentation.
- Les abréviations, nombres, URL et langues différentes constituent des cas particuliers.
- Les règles personnalisées doivent être testées et adaptées au corpus.